<div class="alert alert-block alert-success">
# Flow of the code

Install Packages → Import Libraries → Load Data → Clean Text → Split Data
        ↓
Load Pre-trained Model → Create Datasets → Train Model → Save Model
        ↓
Create Prediction Function → Test Model → Deploy
    
</div>

## 1: Install Required Packages

In [ ]:
# Install required packages (run once)
import subprocess
import sys

packages = ['torch', 'transformers', 'scikit-learn', 'pandas', 'tqdm']
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])


## 2: Import Libraries

In [ ]:

#Import libraries
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from tqdm import tqdm
import requests
import zipfile
import io
import warnings

from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')

print("All imports successful!")

## 3.Load Dataset from UCI

In [16]:
# Download dataset from UCI directly
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

print("Downloading SMS Spam Collection dataset...")
response = requests.get(url)
response.raise_for_status()

# Extract the zip file
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open('SMSSpamCollection') as f:
        data = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'], encoding='utf-8')

print(f"Dataset loaded successfully!")
print(f"Shape: {data.shape}")
print(f"\nFirst 5 rows:")
print(data.head())

Dataset loaded successfully!
Shape: (5572, 2)

First 5 rows:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


## 4: Explore Dataset

In [17]:
# Explore dataset
print("\nDataset Info:")
print(f"Total messages: {len(data)}")
print(f"Spam messages: {(data['label'] == 'spam').sum()}")
print(f"Ham messages: {(data['label'] == 'ham').sum()}")
print(f"\nLabel distribution:")
print(data['label'].value_counts())

# Map labels to numbers
data['label_num'] = data['label'].map({'ham': 0, 'spam': 1})


Dataset Info:
Total messages: 5572
Spam messages: 747
Ham messages: 4825

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


## 5: Text Preprocessing Function

In [20]:
# Text preprocessing function
def clean_text(text):
    """Clean SMS text"""
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove numbers but keep words
    text = re.sub(r'\d+', '', text)
    # Remove special characters
    text = re.sub(r'[^\w\s]', '', text)
    # Remove extra spaces
    text = ' '.join(text.split())
    return text

# Apply preprocessing
print("Cleaning messages...")
data['clean_message'] = data['message'].apply(clean_text)

print("Sample before cleaning:")
print(data['message'].iloc[0])
print("\nSample after cleaning:")
print(data['clean_message'].iloc[0])

Cleaning messages...
Sample before cleaning:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

Sample after cleaning:
go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat


## 6: Split Dataset

In [ ]:
# Split dataset - Convert to numpy arrays explicitly
X = np.array(data['clean_message'].tolist())
y = np.array(data['label_num'].tolist())

print(f"X type: {type(X)}")
print(f"y type: {type(y)}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# First split: 80% train+val, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: split train+val into 80% train, 20% val (64% train, 16% val overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)
#random_state=42: Ensures reproducible splits (same random selection each run)
#stratify=y: Maintains the same proportion of spam/ham in both sets

print(f"\n✅ Splitting successful!")
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining spam ratio: {y_train.mean():.2%}")
print(f"Validation spam ratio: {y_val.mean():.2%}")
print(f"Test spam ratio: {y_test.mean():.2%}")

X type: <class 'numpy.ndarray'>
y type: <class 'numpy.ndarray'>
X shape: (5572,)
y shape: (5572,)

✅ Splitting successful!
Training set size: 3565
Validation set size: 892
Test set size: 1115

Training spam ratio: 13.41%
Validation spam ratio: 13.45%
Test spam ratio: 13.36%


## 7: Load DistilBERT Model

In [25]:
# Load DistilBERT model
print("Loading DistilBERT model...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"✅ Model loaded: {model_name}")
print(f"Model type: DistilBERT")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading DistilBERT model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1983.06it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded: distilbert-base-uncased
Model type: DistilBERT
Number of parameters: 66,955,010


## 8: Create Dataset Class

In [26]:
# Define SpamDataset Class
class SpamDataset(Dataset):
    """Custom Dataset for SMS Spam Classification"""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenize the text
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("SpamDataset class defined successfully!")

SpamDataset class defined successfully!


## 9: Create Datasets

In [27]:
# Create datasets
train_dataset = SpamDataset(X_train, y_train, tokenizer)
val_dataset = SpamDataset(X_val, y_val, tokenizer)
test_dataset = SpamDataset(X_test, y_test, tokenizer)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Training dataset size: 3565
Validation dataset size: 892
Test dataset size: 1115


## 10: Training Setup

In [28]:
# Training setup
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

Using device: cpu


## 11: Train Model

In [29]:
# Training loop
print("\n" + "="*50)
print("Starting Training")
print("="*50)

for epoch in range(3):  # Train for 3 epochs
    # Training phase
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/3 - Training")
    
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels
        )
        
        loss = outputs.loss
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/3 - Validation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids, 
                attention_mask=attention_mask, 
                labels=labels
            )
            
            val_loss += outputs.loss.item()
            predictions = torch.argmax(outputs.logits, dim=-1)
            val_correct += (predictions == labels).sum().item()
            val_total += len(labels)
    
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = val_correct / val_total
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f}")
    print(f"  Val Accuracy: {val_accuracy:.4f}")
    print("-"*40)

print("\n✅ Training completed!")


Starting Training


Epoch 1/3 - Validation: 100%|██████████| 112/112 [02:15<00:00,  1.21s/it]



Epoch 1 Summary:
  Train Loss: 0.1070
  Val Loss: 0.0839
  Val Accuracy: 0.9753
----------------------------------------


Epoch 2/3 - Validation: 100%|██████████| 112/112 [02:18<00:00,  1.23s/it]



Epoch 2 Summary:
  Train Loss: 0.0342
  Val Loss: 0.0807
  Val Accuracy: 0.9821
----------------------------------------


Epoch 3/3 - Validation: 100%|██████████| 112/112 [02:10<00:00,  1.17s/it]


Epoch 3 Summary:
  Train Loss: 0.0134
  Val Loss: 0.0723
  Val Accuracy: 0.9854
----------------------------------------

✅ Training completed!


## 12: Evaluate on Test Set

In [30]:
# Evaluate on test set
test_loader = DataLoader(test_dataset, batch_size=8)
model.eval()
test_predictions = []
test_true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating on test set"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        test_predictions.extend(predictions.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

# Calculate metrics
test_accuracy = accuracy_score(test_true_labels, test_predictions)
print(f"\n📊 Test Set Results:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(test_true_labels, test_predictions, target_names=['Ham', 'Spam']))
print(f"\nConfusion Matrix:")
print(confusion_matrix(test_true_labels, test_predictions))

Evaluating on test set: 100%|██████████| 140/140 [03:00<00:00,  1.29s/it]


📊 Test Set Results:
Accuracy: 0.9857

Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       966
        Spam       0.94      0.95      0.95       149

    accuracy                           0.99      1115
   macro avg       0.97      0.97      0.97      1115
weighted avg       0.99      0.99      0.99      1115


Confusion Matrix:
[[957   9]
 [  7 142]]


## 13: Save Model

In [31]:
# Save the model
model_save_path = "./spam_detector_distilbert"
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✅ Model saved to {model_save_path}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.16it/s]

✅ Model saved to ./spam_detector_distilbert


## 14: Create Prediction Function

In [32]:
# Prediction function
def predict_spam(message):
    """Predict if a message is spam"""
    # Clean message
    cleaned = clean_text(message)
    
    # Tokenize
    inputs = tokenizer(
        cleaned,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    
    return {
        'message': message,
        'is_spam': pred == 1,
        'confidence': probs[0][pred].item(),
        'spam_probability': probs[0][1].item(),
        'ham_probability': probs[0][0].item()
    }

print("Prediction function created successfully!")

Prediction function created successfully!


## 15: Test the Model

In [33]:
# Test with sample messages
test_messages = [
    "Hey, want to grab lunch tomorrow?",
    "FREE iPhone! Click here to claim your prize!",
    "Your meeting is at 3 PM in room 204.",
    "URGENT: Your account has been suspended! Verify now!",
    "Can you send me the presentation slides?",
    "Congratulations! You've won a $1000 gift card!",
    "Call 1800-FREE-MONEY to claim your prize!",
    "Reminder: Doctor appointment tomorrow at 10 AM"
]

print("\n" + "="*60)
print("🔍 SPAM DETECTOR TEST RESULTS")
print("="*60)

for msg in test_messages:
    result = predict_spam(msg)
    status = "🔴 SPAM" if result['is_spam'] else "🟢 HAM"
    print(f"\n📱 Message: {msg}")
    print(f"   Prediction: {status}")
    print(f"   Confidence: {result['confidence']:.2%}")
    print(f"   Spam probability: {result['spam_probability']:.2%}")
    print(f"   Ham probability: {result['ham_probability']:.2%}")
    print("-"*40)


🔍 SPAM DETECTOR TEST RESULTS

📱 Message: Hey, want to grab lunch tomorrow?
   Prediction: 🟢 HAM
   Confidence: 99.94%
   Spam probability: 0.06%
   Ham probability: 99.94%
----------------------------------------

📱 Message: FREE iPhone! Click here to claim your prize!
   Prediction: 🔴 SPAM
   Confidence: 99.64%
   Spam probability: 99.64%
   Ham probability: 0.36%
----------------------------------------

📱 Message: Your meeting is at 3 PM in room 204.
   Prediction: 🟢 HAM
   Confidence: 99.83%
   Spam probability: 0.17%
   Ham probability: 99.83%
----------------------------------------

📱 Message: URGENT: Your account has been suspended! Verify now!
   Prediction: 🔴 SPAM
   Confidence: 99.48%
   Spam probability: 99.48%
   Ham probability: 0.52%
----------------------------------------

📱 Message: Can you send me the presentation slides?
   Prediction: 🟢 HAM
   Confidence: 99.68%
   Spam probability: 0.32%
   Ham probability: 99.68%
----------------------------------------

📱 Messa

## 16: Interactive Test Function 

In [34]:
# Interactive testing function
def interactive_test():
    """Interactive mode to test custom messages"""
    print("\n" + "="*60)
    print("🔍 INTERACTIVE SPAM DETECTOR")
    print("="*60)
    print("Enter your message (or 'quit' to exit):")
    
    while True:
        message = input("\n📱 Your message: ")
        if message.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        result = predict_spam(message)
        status = "🔴 SPAM" if result['is_spam'] else "🟢 HAM"
        
        print(f"\nResult: {status}")
        print(f"Confidence: {result['confidence']:.2%}")
        if result['is_spam']:
            print(f"⚠️ This message appears to be SPAM!")
        else:
            print(f"✅ This message appears to be legitimate (HAM).")

# Uncomment to run interactive test
# interactive_test()

## 17: Load Saved Model

In [35]:
# Load saved model (use this in a new session)
def load_saved_model(model_path="./spam_detector_distilbert"):
    """Load the saved model and tokenizer"""
    loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loaded_model = loaded_model.to(device)
    
    print(f"✅ Model loaded from {model_path}")
    return loaded_model, loaded_tokenizer, device

# Example usage:
# loaded_model, loaded_tokenizer, device = load_saved_model()